# Quy trình thu thập dữ liệu tốc độ cao giai đoạn 1 trên Google Colab

Sổ tay thực thi thu thập mã nguồn LaTeX theo từng lô 100 bài báo cho tập 20.000 định danh arXiv của Nhóm 10:
1. **Thư mục làm việc cục bộ (`/content/fast_crawling_workspace`)**: Ghi dữ liệu trực tiếp trên ổ đĩa cục bộ của máy ảo để tránh độ trễ ghi tệp nhỏ trên Google Drive.
2. **Thư mục giải nén tạm trên bộ nhớ RAM (`/dev/shm/arxiv_scratch`)**: Giải nén tệp nguồn `.tar.gz` và lọc tệp `.tex`, `.bib` trong bộ nhớ RAM trước khi lưu vào thư mục phiên bản.
3. **Nạp trước siêu dữ liệu từ bộ dữ liệu Kaggle (`arxiv-metadata-oai-snapshot.json`)**: Trích xuất danh sách phiên bản (`v1`, `v2`, ...) và ngày nộp của từng phiên bản từ tệp cục bộ thay vì truy vấn tuần tự qua mạng.
4. **Lưu trạng thái và khôi phục tiến trình**: Ghi nhận tiến độ từng bài báo vào `manifest.json` và đồng bộ tệp nén `cohort_XXXX_corpus.tar.gz` về Google Drive sau mỗi lượt chạy hoặc khi dừng giữa chừng.
5. **Kiểm soát dừng có kỷ luật**: Dừng ngay lập tức toàn bộ chuỗi lô khi nhận mã lỗi 403 hoặc cạn ngân sách chờ 429.

In [ ]:
# Bước 1: Kết nối Google Drive và chuẩn bị mã nguồn
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/Arxiv-Project-Group10')
if not REPO_DIR.exists():
    !git clone -b feat/m1-t02-colab-resume https://github.com/Toan-ten/Arxiv-Project-Group10.git /content/Arxiv-Project-Group10
else:
    %cd /content/Arxiv-Project-Group10
    !git fetch origin feat/m1-t02-colab-resume
    !git checkout feat/m1-t02-colab-resume
    !git reset --hard origin/feat/m1-t02-colab-resume

%cd /content/Arxiv-Project-Group10
!pip install -q -r requirements.txt
print('Đã thiết lập môi trường và mã nguồn tại:', REPO_DIR)

In [ ]:
# Bước 2: Nạp cấu hình Kaggle API an toàn từ biến môi trường hoặc Colab Secrets
import os
from pathlib import Path

kaggle_token = os.environ.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    try:
        from google.colab import userdata
        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass

if kaggle_token:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    (kaggle_dir / "access_token").write_text(kaggle_token.strip() + "
")
    (kaggle_dir / "access_token").chmod(0o600)
    print("Đã thiết lập Kaggle access token từ cấu hình bảo mật.")
else:
    print("Chưa cấu hình KAGGLE_API_TOKEN. Bỏ qua bước tải tự động từ Kaggle API.")

snapshot_file = Path("arxiv-metadata-oai-snapshot.json")
if not snapshot_file.exists() and kaggle_token:
    print("Đang tải và giải nén bộ dữ liệu Cornell-University/arxiv từ Kaggle...")
    !kaggle datasets download -d Cornell-University/arxiv
    !unzip -q -o arxiv.zip
    print("Đã giải nén tệp arxiv-metadata-oai-snapshot.json.")
elif snapshot_file.exists():
    print("Tệp arxiv-metadata-oai-snapshot.json đã có sẵn trên ổ đĩa.")

In [ ]:
# Bước 3: Thực thi thu thập theo từng lô (100 định danh / lô)
# Phân bổ 200 lô dữ liệu:
# - Lô 1..50:    24120010 (Trần Văn Tấn Khôi)
# - Lô 51..100:  24120035 (Lê Thành Đạt)
# - Lô 101..150: 24120232 (Nguyễn Văn Toàn)
# - Lô 151..200: 24120260 (Trương Tuấn Anh)
from pathlib import Path
from src.fast_production import FastProductionRunner

LOCAL_ROOT = '/content/fast_crawling_workspace'
DRIVE_ROOT = '/content/drive/MyDrive/Group10_M1_Production'

runner = FastProductionRunner(
    local_root=LOCAL_ROOT,
    drive_root=DRIVE_ROOT,
    retry_429=True,
    retry_wait_budget=300.0,
    timeout=30.0,
)

COHORT_LIST = [151]

for cohort_num in COHORT_LIST:
    restored = runner.restore_cohort(cohort_num)
    if restored > 0:
        print(f'Đã khôi phục {restored} thư mục bài báo của lô {cohort_num} từ Google Drive.')

snapshot_path = Path('arxiv-metadata-oai-snapshot.json')
discovery_interrupted = False
if snapshot_path.exists():
    print(f'Đang đối chiếu siêu dữ liệu từ bộ dữ liệu Kaggle cho danh sách lô {COHORT_LIST}...')
    res = runner.preload_kaggle_discovery(snapshot_path, COHORT_LIST)
    if res.get('interrupted'):
        discovery_interrupted = True
    else:
        print(f'Đã ghi nhận siêu dữ liệu cho {res["found_papers"]}/{res["target_papers"]} bài báo trong {res.get("elapsed_seconds", 0.0):.2f}s (RAM đỉnh: {res.get("peak_ram_mb", 0.0):.1f} MB).')
        print(f'Báo cáo đo lường bước đối chiếu siêu dữ liệu: {res.get("report_path")}')

if not discovery_interrupted:
    for cohort_num in COHORT_LIST:
        print(f'\nBắt đầu thu thập lô {cohort_num} (100 định danh)...')
        report = runner.run_cohort(cohort_num, resume=True, revision_history=True)
        stats = report.get('fast_runner', {})
        completed = stats.get('completed_papers', 0)
        elapsed = stats.get('elapsed_seconds', 0.0)
        if stats.get('interrupted') or completed < 100:
            print(f'Đã lưu trạng thái lô {cohort_num}: hoàn thành {completed}/100 bài báo (thời gian phiên chạy: {elapsed:.1f}s).')
        else:
            print(f'Hoàn tất lô {cohort_num}: hoàn thành {completed}/100 bài báo trong {elapsed:.1f}s.')
            print(f'Thời gian trung bình: {stats.get("avg_seconds_per_paper", 0.0):.2f}s / định danh')
        print(f'Tệp sao lưu trên Google Drive: {stats.get("archive_path")}')
        if stats.get('interrupted') or stats.get('stopped'):
            stop_reason = stats.get('stop_reason')
            if stop_reason:
                print(f'Dừng xử lý các lô tiếp theo do điều kiện giới hạn: {stop_reason}.')
            break
